# SU(3) T1⁺⁻ — single A100 master notebook

One notebook, one GPU, everything in order. Host it on GitHub and open with
`https://colab.research.google.com/github/<user>/<repo>/blob/main/su3_a100_master.ipynb`.

**Run order (top to bottom).**
1. **Cell 1** — environment: GPU check, optional Drive mount, locate your
   campaign script. Requirement: `SU3_T1pm_spatial_MC_nextrun2.py` (preferred)
   or `ENGINE_MC_su3_t1pm_spatial_nextrun.py` uploaded to `/content` or present in
   `MyDrive`. This notebook contains everything else.
2. **Cell 2** — the independent PyTorch heatbath implementation, defined
   inline, followed by its exactness gates: G1 (unitarity, over-relaxation
   audit, gauge invariance) and G2 (exact Haar second moments 1/18 at
   β = 0.05). Minutes.
3. **Cell 3** — the two-implementation cross-check, both streams on this A100:
   your CuPy Metropolis chain and the PyTorch Kennedy–Pendleton heatbath chain
   at identical β = 5.99, 8³×16, 400 configs each, independent seeds, each
   using its own measurement kernel, compared with shared statistics. Prints
   the |Δ|/σ gate table and saves both result JSONs. ≈10–30 min total.
4. **Cell 4** — stage-1 production, one ensemble per session:
   `ENSEMBLE_INDEX 1 → β 5.99 (18³×18)`, `2 → 6.0625 (20³×20)`,
   `3 → 6.235 (26³×26)`.
5. **Cell 5** — combine instructions for the continuum refit.

**Honesty flags.** `nextrun` (non-2) writes JSON only at completion — no
mid-run checkpoint — so calibrate wall time on index 1 before attempting
index 3 (26³) and use `nextrun2` for the long ensembles. The doubly-wound P2
channel is measured by the PyTorch stream with the ImTr(W²)/3 convention;
its cross-check against `nextrun2` waits on the normalisation confirmation
flagged below, so Cell 3 gates the plaquette and P channels only.

In [ ]:
# Cell 1 -- environment and campaign-script discovery
import glob, os, subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout or "no nvidia-smi (CPU mode)")
OUTDIR = "/content"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTDIR = "/content/drive/MyDrive/su3_stage1"
    os.makedirs(OUTDIR, exist_ok=True)
except Exception as e:
    OUTDIR = os.getcwd()
    print("Drive not mounted (fine outside Colab):", e)
print("outputs ->", OUTDIR)

def find(names):
    for n in names:
        hits = glob.glob(f"/content/{n}") + glob.glob(f"/content/drive/MyDrive/**/{n}", recursive=True) + glob.glob(n)
        if hits:
            return hits[0]
    return None

SCRIPT_PROD = find(["SU3_T1pm_spatial_MC_nextrun2.py", "ENGINE_MC_su3_t1pm_spatial_nextrun.py"])
SCRIPT_REF = find(["ENGINE_MC_su3_t1pm_spatial_nextrun.py", "SU3_T1pm_spatial_MC_nextrun2.py"])
if SCRIPT_PROD is None:
    raise FileNotFoundError("Upload SU3_T1pm_spatial_MC_nextrun2.py (or nextrun.py), then rerun.")
print("production script:", SCRIPT_PROD)
print("reference-run script:", SCRIPT_REF)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"], check=False)
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
outputs -> /content/drive/MyDrive/su3_stage1
production script: /content/ENGINE_MC_su3_t1pm_spatial_nextrun.py
reference-run script: /content/ENGINE_MC_su3_t1pm_spatial_nextrun.py


In [ ]:
#!/usr/bin/env python3
"""
su3_t1pm_pytorch_port_v1.py -- SIM-2: independent-implementation MC stream.

PyTorch port of the SU(3) T1^{+-} spatial pipeline. Runs unmodified on CUDA
(A100), ROCm (7900 XTX: torch.cuda.is_available() is True on ROCm builds), and
CPU. Zero arguments, zero environment variables, zero edits required:

  * On every run: gate block G1 (exactness gates) and G2 (Haar-limit
    distributional anchor at beta = 0.05) execute on a small lattice.
  * If a GPU is present: gate block G3 executes -- a beta = 5.99 cross-check
    ensemble at L = 8, Nt = 16 whose observables are dumped to
    ./su3_port_g3_b599.json for comparison against the CuPy stream at the
    identical volume (nextrun profile 'pilot' geometry).
  * Production stream (L = 18, Nt = 18, beta = 5.99, stage-1 volume) arms ONLY
    if a file named RUN_PRODUCTION exists in the working directory
    (create it with:  touch RUN_PRODUCTION).  No variables to edit.

PROVENANCE INDEPENDENCE. The reference chain (ENGINE_MC_su3_t1pm_spatial_nextrun.py)
uses checkerboard SU(2)-subgroup METROPOLIS + over-relaxation. This port uses
checkerboard Cabibbo-Marinari KENNEDY-PENDLETON HEATBATH + over-relaxation: a
different exact algorithm targeting the same Wilson measure. Agreement at G3
therefore certifies the equilibrium distribution and the measurement kernels,
not merely a code transliteration.

MATCHED CONVENTIONS (verified against the reference source):
  action        S = -(beta/3) sum_p Re Tr U_p   (Metropolis delta uses beta/3)
  links         complex64, layout U[Nt, L, L, L, mu, 3, 3], mu = 0 is time
  shift         shift(a, mu, +1)[x] = a[x + mu_hat]   (torch.roll, -1)
  staple        V(mu) = sum_{nu != mu} forward + backward, exactly as reference
  T1 operator   O_i(t) = sum_x Im Tr W_{jk}(x, t) / 3 / sqrt(L^3),
                cyclic planes x->(y,z), y->(z,x), z->(x,y), path
                W = U_j shift_j(U_k) dagger(shift_k(U_j)) dagger(U_k)
  P2 operator   Im Tr (W^2) / 3, same pipeline.  FLAG: nextrun2's doubly-wound
                normalisation was not visible in the project copy; confirm the
                1/3 (not 1/6) convention against nextrun2 before comparing the
                P2 channel at G3.  The P channel comparison is unconditional.
  correlator    per-config C[t] averaged over the 3 components and Nt origins,
                after subtracting one GLOBAL mean over (cfg, component, time)
  SU(2) embed   G(q) = q0*1 + i(q1 s1 + q2 s2 + q3 s3) on rows (i, j),
                identical to reference _left_su2_rows
  reunitarize   column Gram-Schmidt, third column = conj(a x b), as reference

FP PLAN. Links complex64 (matches reference; sidesteps RDNA3's 1/16 FP64
rate); all observable accumulators float64; reunitarization every
REUNITARIZE_EVERY cycles with unitarity audit.

Gates:
  G1a  cold-start plaquette == 1
  G1b  unitarity and determinant errors after thermalisation
  G1c  over-relaxation preserves the local action (fp32 tolerance)
  G1d  gauge invariance: random SU(3) gauge transform leaves the plaquette
       and all T1/P2 operators invariant (independent correctness gate the
       two implementations cannot share a bug through)
  G2   Haar anchor at beta = 0.05: <(1/3)ReTr P> ~ beta/18, and the EXACT
       Haar second moments <((1/3)ReTr)^2> = <((1/3)ImTr)^2> = 1/18
  G3   cross-implementation observables at beta = 5.99 (GPU only)
"""
from __future__ import annotations

import json
import math
import os
import time

import numpy as np
import torch

torch.manual_seed(20260815)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CDTYPE = torch.complex64
RDTYPE = torch.float32
REUNITARIZE_EVERY = 25
PAIRS = ((0, 1), (0, 2), (1, 2))

PASS = FAIL = 0
def gate(name: str, ok: bool, detail: str = "") -> None:
    global PASS, FAIL
    print(("[PASS] " if ok else "[FAIL] ") + name + (f" :: {detail}" if detail else ""))
    PASS, FAIL = PASS + ok, FAIL + (not ok)


# =============================================================================
# SU(2)-subgroup algebra (quaternion conventions matched to the reference)
# =============================================================================

def quat_mul(a, b):
    """(a0, a_vec)(b0, b_vec) = (a0 b0 - a.b, a0 b_vec + b0 a_vec + a x b)."""
    a0, a1, a2, a3 = a
    b0, b1, b2, b3 = b
    return (
        a0 * b0 - a1 * b1 - a2 * b2 - a3 * b3,
        a0 * b1 + b0 * a1 + a2 * b3 - a3 * b2,
        a0 * b2 + b0 * a2 + a3 * b1 - a1 * b3,
        a0 * b3 + b0 * a3 + a1 * b2 - a2 * b1,
    )


def block_quaternion(W, pair):
    """SU(2)-projected block of the 3x3 matrix W at rows/cols (i, j).

    Returns (k, p) with k >= 0 and p a unit quaternion such that
    Re Tr_2( G(q) W_block ) = 2 k (q . p) for any unit quaternion q.
    """
    i, j = pair
    wii, wij = W[..., i, i], W[..., i, j]
    wji, wjj = W[..., j, i], W[..., j, j]
    a0 = 0.5 * (wii + wjj).real
    a1 = 0.5 * (wij + wji).imag
    a2 = 0.5 * (wij - wji).real
    a3 = 0.5 * (wii - wjj).imag
    p0, p1, p2, p3 = a0, -a1, -a2, -a3
    k = torch.sqrt(torch.clamp(p0 * p0 + p1 * p1 + p2 * p2 + p3 * p3, min=1e-30))
    ok = k > 1e-12
    one, zero = torch.ones_like(k), torch.zeros_like(k)
    p = (torch.where(ok, p0 / k, one),
         torch.where(ok, p1 / k, zero),
         torch.where(ok, p2 / k, zero),
         torch.where(ok, p3 / k, zero))
    return torch.where(ok, k, torch.zeros_like(k)), p


def apply_left_su2(U, pair, q):
    """U <- G(q) U with G(q) = q0 + i q.sigma embedded on rows (i, j)."""
    i, j = pair
    q0, q1, q2, q3 = q
    a00 = torch.complex(q0, q3).to(CDTYPE)
    a01 = torch.complex(q2, q1).to(CDTYPE)
    a10 = torch.complex(-q2, q1).to(CDTYPE)
    a11 = torch.complex(q0, -q3).to(CDTYPE)
    out = U.clone()
    Ui, Uj = U[..., i, :], U[..., j, :]
    out[..., i, :] = a00[..., None] * Ui + a01[..., None] * Uj
    out[..., j, :] = a10[..., None] * Ui + a11[..., None] * Uj
    return out


def sample_x0(alpha, gen_shape):
    """Sample x0 in [-1, 1] with density ~ sqrt(1 - x0^2) exp(alpha x0).

    Kennedy-Pendleton for alpha >= 0.5; exponential-envelope rejection for
    alpha < 0.5 (covers the Haar limit alpha -> 0 exactly). Fully vectorised
    masked retry; float64 internals.
    """
    alpha = alpha.to(torch.float64)
    x0 = torch.zeros(gen_shape, dtype=torch.float64, device=alpha.device)
    active = torch.ones(gen_shape, dtype=torch.bool, device=alpha.device)
    use_kp = alpha >= 0.5
    a_safe = torch.clamp(alpha, min=1e-12)
    for _ in range(500):
        if not bool(active.any()):
            break
        r1 = torch.rand(gen_shape, dtype=torch.float64, device=alpha.device).clamp_(1e-12, 1.0)
        r2 = torch.rand(gen_shape, dtype=torch.float64, device=alpha.device)
        r3 = torch.rand(gen_shape, dtype=torch.float64, device=alpha.device).clamp_(1e-12, 1.0)
        r4 = torch.rand(gen_shape, dtype=torch.float64, device=alpha.device)
        # --- Kennedy-Pendleton branch ---
        lam2 = -(torch.log(r1) + torch.cos(2.0 * math.pi * r2) ** 2 * torch.log(r3)) / (2.0 * a_safe)
        kp_accept = (r4 * r4) <= torch.clamp(1.0 - lam2, min=0.0)
        kp_x0 = 1.0 - 2.0 * lam2
        # --- exponential-envelope branch (small alpha) ---
        small = a_safe < 1e-9
        span = torch.exp(alpha) - torch.exp(-alpha)
        prop = torch.where(
            small,
            2.0 * r1 - 1.0,
            torch.log(torch.clamp(torch.exp(-alpha) + r1 * span, min=1e-300)) / a_safe,
        ).clamp(-1.0, 1.0)
        env_accept = r4 <= torch.sqrt(torch.clamp(1.0 - prop * prop, min=0.0))
        cand = torch.where(use_kp, kp_x0, prop)
        acc = active & torch.where(use_kp, kp_accept, env_accept)
        x0 = torch.where(acc, cand, x0)
        active = active & ~acc
    if bool(active.any()):
        raise RuntimeError("SU(2) heatbath sampler failed to converge")
    return x0.clamp(-1.0, 1.0)


def sample_su2_heatbath(alpha):
    """Return unit quaternion x with P(x) ~ exp(alpha * x0) d(Haar SU(2))."""
    shape = alpha.shape
    x0 = sample_x0(alpha, shape)
    vec = torch.randn(shape + (3,), dtype=torch.float64, device=alpha.device)
    vec = vec / torch.sqrt(torch.clamp((vec * vec).sum(-1, keepdim=True), min=1e-30))
    rho = torch.sqrt(torch.clamp(1.0 - x0 * x0, min=0.0))
    return (x0.to(RDTYPE),
            (rho * vec[..., 0]).to(RDTYPE),
            (rho * vec[..., 1]).to(RDTYPE),
            (rho * vec[..., 2]).to(RDTYPE))


# =============================================================================
# Lattice
# =============================================================================

class Su3Lattice:
    def __init__(self, beta: float, L: int, Nt: int):
        self.beta, self.L, self.Nt = float(beta), int(L), int(Nt)
        if L % 2 or Nt % 2:
            raise ValueError("checkerboard updates require even L and Nt")
        self.shape = (Nt, L, L, L)
        eye = torch.eye(3, dtype=CDTYPE, device=DEVICE)
        self.U = eye.expand(self.shape + (4, 3, 3)).clone()
        coords = torch.stack(torch.meshgrid(
            *[torch.arange(n, device=DEVICE) for n in self.shape], indexing="ij"))
        self.parity = (coords.sum(0) & 1).bool()

    # -- primitives ----------------------------------------------------------
    @staticmethod
    def dagger(a):
        return a.transpose(-1, -2).conj()

    def shift(self, a, mu: int, amount: int):
        return torch.roll(a, shifts=-int(amount), dims=int(mu))

    def staple(self, mu: int):
        U_mu = self.U[..., mu, :, :]
        V = torch.zeros_like(U_mu)
        for nu in range(4):
            if nu == mu:
                continue
            U_nu = self.U[..., nu, :, :]
            forward = (self.shift(U_nu, mu, 1)
                       @ self.dagger(self.shift(U_mu, nu, 1))
                       @ self.dagger(U_nu))
            U_nu_m = self.shift(U_nu, nu, -1)
            backward = (self.dagger(self.shift(U_nu_m, mu, 1))
                        @ self.dagger(self.shift(U_mu, nu, -1))
                        @ U_nu_m)
            V = V + forward + backward
        return V

    # -- updates -------------------------------------------------------------
    def heatbath_sweep(self) -> None:
        for mu in range(4):
            for parity_value in (0, 1):
                mask = self.parity == parity_value
                V = self.staple(mu)
                Umu = self.U[..., mu, :, :]
                for pair in PAIRS:
                    W = Umu @ V
                    k, p = block_quaternion(W, pair)
                    alpha = (2.0 * self.beta / 3.0) * k
                    x = sample_su2_heatbath(alpha)
                    g = quat_mul(x, p)
                    proposed = apply_left_su2(Umu, pair, g)
                    Umu = torch.where(mask[..., None, None], proposed, Umu)
                self.U[..., mu, :, :] = Umu

    def overrelax_sweep(self, audit: bool = False) -> float:
        max_err = torch.zeros((), dtype=torch.float64, device=DEVICE)
        for mu in range(4):
            for parity_value in (0, 1):
                mask = self.parity == parity_value
                V = self.staple(mu)
                Umu = self.U[..., mu, :, :]
                for pair in PAIRS:
                    W = Umu @ V
                    _, h = block_quaternion(W, pair)
                    q = (2.0 * h[0] * h[0] - 1.0,
                         2.0 * h[0] * h[1], 2.0 * h[0] * h[2], 2.0 * h[0] * h[3])
                    proposed = apply_left_su2(Umu, pair, q)
                    if audit:
                        old = torch.einsum("...ij,...ji->...", Umu, V).real
                        new = torch.einsum("...ij,...ji->...", proposed, V).real
                        err = torch.where(mask, (new - old).abs(), torch.zeros_like(old))
                        max_err = torch.maximum(max_err, err.max().double())
                    Umu = torch.where(mask[..., None, None], proposed, Umu)
                self.U[..., mu, :, :] = Umu
        return float(max_err)

    def cycle(self, n_or: int = 2) -> None:
        self.heatbath_sweep()
        for _ in range(n_or):
            self.overrelax_sweep()

    # -- exactness maintenance ------------------------------------------------
    def reunitarize_matrix(self, M):
        a = M[..., :, 0]
        a = a / torch.sqrt(torch.clamp((a.conj() * a).sum(-1, keepdim=True).real, min=1e-30))
        b = M[..., :, 1]
        b = b - a * (a.conj() * b).sum(-1, keepdim=True)
        b = b / torch.sqrt(torch.clamp((b.conj() * b).sum(-1, keepdim=True).real, min=1e-30))
        c = torch.stack((a[..., 1] * b[..., 2] - a[..., 2] * b[..., 1],
                         a[..., 2] * b[..., 0] - a[..., 0] * b[..., 2],
                         a[..., 0] * b[..., 1] - a[..., 1] * b[..., 0]), dim=-1).conj()
        return torch.stack((a, b, c), dim=-1).to(CDTYPE)

    def reunitarize(self) -> None:
        self.U = self.reunitarize_matrix(self.U)

    def group_errors(self):
        eye = torch.eye(3, dtype=CDTYPE, device=DEVICE)
        uu = self.dagger(self.U) @ self.U - eye
        unit = torch.sqrt((uu.real ** 2 + uu.imag ** 2).sum((-2, -1))).max()
        det = torch.linalg.det(self.U.to(torch.complex128))
        return float(unit), float((det - 1.0).abs().max())

    # -- observables ----------------------------------------------------------
    def plaquette_field(self, mu: int, nu: int):
        U_mu, U_nu = self.U[..., mu, :, :], self.U[..., nu, :, :]
        return (U_mu @ self.shift(U_nu, mu, 1)
                @ self.dagger(self.shift(U_mu, nu, 1)) @ self.dagger(U_nu))

    def plaquette_moments(self):
        """(mean ReTr/3, mean (ReTr/3)^2, mean ImTr/3, mean (ImTr/3)^2)."""
        s = torch.zeros(4, dtype=torch.float64, device=DEVICE)
        n = 0
        for mu in range(4):
            for nu in range(mu + 1, 4):
                tr = torch.einsum("...ii->...", self.plaquette_field(mu, nu)) / 3.0
                s[0] += tr.real.double().sum(); s[1] += (tr.real.double() ** 2).sum()
                s[2] += tr.imag.double().sum(); s[3] += (tr.imag.double() ** 2).sum()
                n += tr.numel()
        return tuple(float(v) for v in (s / n))

    def plaquette(self) -> float:
        return self.plaquette_moments()[0]

    def ape_step(self, spatial):
        out = spatial.clone()
        for i in range(3):
            axis_i = i + 1
            Ui = spatial[..., i, :, :]
            st = torch.zeros_like(Ui)
            for j in range(3):
                if i == j:
                    continue
                axis_j = j + 1
                Uj = spatial[..., j, :, :]
                fwd = Uj @ self.shift(Ui, axis_j, 1) @ self.dagger(self.shift(Uj, axis_i, 1))
                Uj_m = self.shift(Uj, axis_j, -1)
                bwd = self.dagger(Uj_m) @ self.shift(Ui, axis_j, -1) @ self.shift(Uj_m, axis_i, 1)
                st = st + fwd + bwd
            out[..., i, :, :] = self.reunitarize_matrix(0.5 * Ui + 0.125 * st)
        return out

    def t1_operators(self, spatial) -> np.ndarray:
        """[n_op = 2 (P, P2), 3 components, Nt], reference normalisation."""
        vol = math.sqrt(self.L ** 3)
        comps_P, comps_P2 = [], []
        for j, k in ((1, 2), (2, 0), (0, 1)):     # cyclic planes for i = x, y, z
            Uj, Uk = spatial[..., j, :, :], spatial[..., k, :, :]
            W = (Uj @ self.shift(Uk, j + 1, 1)
                 @ self.dagger(self.shift(Uj, k + 1, 1)) @ self.dagger(Uk))
            trP = torch.einsum("...ii->...", W).imag.double() / 3.0
            trP2 = torch.einsum("...ii->...", W @ W).imag.double() / 3.0
            comps_P.append((trP.sum((1, 2, 3)) / vol).cpu().numpy())
            comps_P2.append((trP2.sum((1, 2, 3)) / vol).cpu().numpy())
        return np.stack([np.stack(comps_P), np.stack(comps_P2)]).astype(np.float64)

    def measure(self, ape_levels=(0,)) -> np.ndarray:
        spatial = self.U[..., 1:4, :, :].clone()
        blocks, level = [], 0
        for target in sorted(set(ape_levels)):
            while level < target:
                spatial = self.ape_step(spatial)
                level += 1
            blocks.append(self.t1_operators(spatial))
        return np.concatenate(blocks, axis=0)      # [n_level*2, 3, Nt]

    def random_gauge_transform(self) -> None:
        g = torch.eye(3, dtype=CDTYPE, device=DEVICE).expand(self.shape + (3, 3)).clone()
        for _ in range(2):
            for pair in PAIRS:
                q = torch.randn(self.shape + (4,), dtype=torch.float64, device=DEVICE)
                q = q / torch.sqrt(torch.clamp((q * q).sum(-1, keepdim=True), min=1e-30))
                g = apply_left_su2(g, pair, tuple(q[..., a].to(RDTYPE) for a in range(4)))
        g = self.reunitarize_matrix(g)
        for mu in range(4):
            self.U[..., mu, :, :] = (
                g @ self.U[..., mu, :, :] @ self.dagger(self.shift(g, mu, 1)))


# =============================================================================
# Correlators and statistics (reference conventions)
# =============================================================================

def correlators(obs: np.ndarray) -> np.ndarray:
    """obs [ncfg, nop, 3, Nt] -> per-config diagonal correlators [ncfg, nop, T+1]."""
    ncfg, nop, ncomp, Nt = obs.shape
    centered = obs - obs.mean(axis=(0, 2, 3), keepdims=True)
    T = Nt // 2
    out = np.empty((ncfg, nop, T + 1))
    for dt in range(T + 1):
        shifted = np.roll(centered, -dt, axis=-1)
        out[:, :, dt] = np.einsum("nact,nact->na", shifted, centered) / float(ncomp * Nt)
    return out

def tau_int(x: np.ndarray) -> float:
    x = np.asarray(x, float)
    if len(x) < 8 or np.var(x) == 0:
        return 0.5
    y = x - x.mean()
    acov = np.correlate(y, y, "full")[len(y) - 1:] / len(y)
    rho, tau = acov / acov[0], 0.5
    for lag in range(1, len(x)):
        if rho[lag] <= 0:
            break
        tau += float(rho[lag])
        if lag >= 5.0 * tau:
            break
    return max(0.5, tau)

def effective_mass(g: np.ndarray, Nt: int, t: int) -> float:
    if g[t] <= 0 or g[t + 1] <= 0:
        return float("nan")
    target = g[t + 1] / g[t]
    lo, hi = 1e-5, 12.0
    f = lambda m: ((math.exp(-m * (t + 1)) + math.exp(-m * (Nt - t - 1)))
                   / (math.exp(-m * t) + math.exp(-m * (Nt - t))) - target)
    if f(lo) * f(hi) > 0:
        return float(-math.log(target))
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        (lo, hi) = (mid, hi) if f(lo) * f(mid) > 0 else (lo, mid)
    return 0.5 * (lo + hi)

def blocked_bootstrap(percfg: np.ndarray, Nt: int, t: int, block: int, nboot: int = 200):
    ncfg = percfg.shape[0]
    nb = max(2, ncfg // block)
    blocks = percfg[: nb * block].reshape(nb, block, -1).mean(axis=1)
    rng = np.random.default_rng(20260815)
    g_mean = percfg.mean(axis=0)
    boots_g, boots_m = [], []
    for _ in range(nboot):
        g = blocks[rng.integers(0, nb, nb)].mean(axis=0)
        boots_g.append(g)
        boots_m.append(effective_mass(g, Nt, t))
    return (g_mean, np.std(np.asarray(boots_g), axis=0),
            effective_mass(g_mean, Nt, t), float(np.nanstd(np.asarray(boots_m))))


# =============================================================================
# Runs
# =============================================================================

def run_ensemble(beta, L, Nt, thermal, ncfg, sep, n_or=2, ape_levels=(0,), label=""):
    lat = Su3Lattice(beta, L, Nt)
    t0 = time.time()
    for c in range(thermal):
        lat.cycle(n_or)
        if (c + 1) % REUNITARIZE_EVERY == 0:
            lat.reunitarize()
    obs, plaqs = [], []
    for c in range(ncfg):
        for _ in range(sep):
            lat.cycle(n_or)
        if (c + 1) % REUNITARIZE_EVERY == 0:
            lat.reunitarize()
        obs.append(lat.measure(ape_levels))
        plaqs.append(lat.plaquette())
    dt = time.time() - t0
    print(f"  [{label}] beta={beta} {L}^3x{Nt}: {thermal}+{ncfg}x{sep} cycles "
          f"in {dt:.1f}s on {DEVICE} ({torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'cpu'})")
    return lat, np.stack(obs), np.asarray(plaqs)


def gate_block_G1_G2():
    print("\n=== G1: exactness gates (beta = 5.99, 4^3 x 4) ===")
    lat = Su3Lattice(5.99, 4, 4)
    gate("G1a cold-start plaquette == 1", abs(lat.plaquette() - 1.0) < 1e-6,
         f"{lat.plaquette():.9f}")
    for _ in range(30):
        lat.cycle(2)
    lat.reunitarize()
    unit, det = lat.group_errors()
    gate("G1b unitarity/det errors < 5e-6 after thermalisation",
         unit < 5e-6 and det < 5e-6, f"unit={unit:.2e} det={det:.2e}")
    err = lat.overrelax_sweep(audit=True)
    gate("G1c over-relaxation preserves local action (fp32)", err < 2e-2, f"max|d|={err:.2e}")
    plaq_before = lat.plaquette()
    O_before = lat.measure((0,))
    lat.random_gauge_transform()
    plaq_after = lat.plaquette()
    O_after = lat.measure((0,))
    scale = max(np.abs(O_before).max(), 1e-6)
    gate("G1d gauge invariance: plaquette", abs(plaq_after - plaq_before) < 5e-5,
         f"|d|={abs(plaq_after - plaq_before):.2e}")
    gate("G1d gauge invariance: T1/P2 operators",
         np.abs(O_after - O_before).max() / scale < 5e-4,
         f"rel={np.abs(O_after - O_before).max() / scale:.2e}")
    sm = lat.plaquette_moments()
    gate("G1e thermalised plaquette in physical band", 0.30 < sm[0] < 0.80, f"{sm[0]:.4f}")

    print("\n=== G2: Haar-limit anchor (beta = 0.05, 4^3 x 4) ===")
    lat = Su3Lattice(0.05, 4, 4)
    for _ in range(60):
        lat.cycle(1)
    m = np.zeros(4)
    ncfg = 300
    for c in range(ncfg):
        lat.cycle(1)
        if (c + 1) % REUNITARIZE_EVERY == 0:
            lat.reunitarize()
        m += np.asarray(lat.plaquette_moments())
    m /= ncfg
    n_plaq = 6 * 4 ** 4 * ncfg
    sig2 = math.sqrt((1.0 / 162.0) / n_plaq)      # exact Haar var of (ReTr/3)^2
    gate("G2a <ReTr/3> ~ beta/18", abs(m[0] - 0.05 / 18.0) < max(6 * math.sqrt((1/18)/n_plaq), 1.5e-3),
         f"{m[0]:+.5f} vs {0.05/18.0:.5f}")
    gate("G2b <(ReTr/3)^2> == 1/18 (exact Haar moment)",
         abs(m[1] - 1.0 / 18.0) < max(6 * sig2, 1e-3), f"{m[1]:.5f} vs {1/18:.5f}")
    gate("G2c <ImTr/3> ~ 0", abs(m[2]) < 1.5e-3, f"{m[2]:+.5f}")
    gate("G2d <(ImTr/3)^2> == 1/18 (exact Haar moment)",
         abs(m[3] - 1.0 / 18.0) < max(6 * sig2, 1e-3), f"{m[3]:.5f} vs {1/18:.5f}")

    print("\n=== SMOKE: correlator pipeline (beta = 5.99, 4^3 x 4) ===")
    lat, obs, _ = run_ensemble(5.99, 4, 4, thermal=10, ncfg=40, sep=2, label="smoke")
    percfg = correlators(obs)
    g, gerr, meff, merr = blocked_bootstrap(percfg[:, 0], 4, 1, block=4, nboot=100)
    gate("SMOKE g_P(0) > 0 and finite pipeline", g[0] > 0 and np.isfinite(g).all(),
         f"g={np.array2string(g, precision=4)}")


def gate_block_G3():
    print("\n=== G3: cross-implementation ensemble (beta = 5.99, 8^3 x 16) ===")
    lat, obs, plaqs = run_ensemble(5.99, 8, 16, thermal=400, ncfg=400, sep=4,
                                   ape_levels=(0, 4), label="G3")
    percfg = correlators(obs)
    unit, det = lat.group_errors()
    gate("G3 unitarity maintained", unit < 1e-4 and det < 1e-4,
         f"unit={unit:.2e} det={det:.2e}")
    tau = tau_int(percfg[:, 0, 0])
    block = max(2, int(math.ceil(2 * tau)))
    result = {"implementation": "pytorch-heatbath-v1",
              "device": str(DEVICE),
              "beta": 5.99, "L": 8, "Nt": 16, "ncfg": 400, "sep": 4,
              "plaquette": float(plaqs.mean()),
              "plaquette_err": float(plaqs.std(ddof=1) / math.sqrt(len(plaqs))),
              "tau_int_gP0": tau, "block": block,
              "channels": {}}
    names = ["P_lvl0", "P2_lvl0", "P_lvl4", "P2_lvl4"]
    for a, name in enumerate(names[: percfg.shape[1]]):
        g, gerr, meff, merr = blocked_bootstrap(percfg[:, a], 16, 1, block)
        result["channels"][name] = {
            "g": g.tolist(), "g_err": gerr.tolist(),
            "meff_t1": meff, "meff_t1_err": merr}
        print(f"  {name}: g(0)={g[0]:.4e}({gerr[0]:.1e})  "
              f"g(1)={g[1]:.4e}({gerr[1]:.1e})  meff(1)={meff:.4f}({merr:.4f})")
    with open("su3_port_g3_b599.json", "w") as f:
        json.dump(result, f, indent=1)
    print("  -> su3_port_g3_b599.json written.")
    print("  COMPARISON PROTOCOL: run the CuPy reference at the identical volume")
    print("  (profile 'pilot' geometry, beta=5.99, L=8, Nt=16, matched ncfg) and gate")
    print("  |Delta|/sigma_combined < 3 on: plaquette, g_P(t) for t<=4, meff_P(1).")
    print("  The P2 channel comparison waits on the nextrun2 normalisation check.")


def production_stream():
    print("\n=== PRODUCTION: beta = 5.99, 18^3 x 18 stage-1 volume, stream 2 ===")
    lat, obs, plaqs = run_ensemble(5.99, 18, 18, thermal=2500, ncfg=7000, sep=1,
                                   n_or=2, ape_levels=(0, 5, 15, 30), label="prod-b599")
    np.savez_compressed("su3_port_prod_b599_stream2.npz", obs=obs, plaquette=plaqs)
    print("  -> su3_port_prod_b599_stream2.npz written (operators [ncfg, nop, 3, Nt]).")


print(f'implementation cell loaded on {DEVICE}')
gate_block_G1_G2()


implementation cell loaded on cuda

=== G1: exactness gates (beta = 5.99, 4^3 x 4) ===
[PASS] G1a cold-start plaquette == 1 :: 1.000000000
[PASS] G1b unitarity/det errors < 5e-6 after thermalisation :: unit=6.11e-07 det=4.58e-07
[PASS] G1c over-relaxation preserves local action (fp32) :: max|d|=4.77e-06
[PASS] G1d gauge invariance: plaquette :: |d|=2.33e-07
[PASS] G1d gauge invariance: T1/P2 operators :: rel=7.29e-07
[PASS] G1e thermalised plaquette in physical band :: 0.6032

=== G2: Haar-limit anchor (beta = 0.05, 4^3 x 4) ===
[PASS] G2a <ReTr/3> ~ beta/18 :: +0.00272 vs 0.00278
[PASS] G2b <(ReTr/3)^2> == 1/18 (exact Haar moment) :: 0.05608 vs 0.05556
[PASS] G2c <ImTr/3> ~ 0 :: +0.00020
[PASS] G2d <(ImTr/3)^2> == 1/18 (exact Haar moment) :: 0.05489 vs 0.05556

=== SMOKE: correlator pipeline (beta = 5.99, 4^3 x 4) ===
  [smoke] beta=5.99 4^3x4: 10+40x2 cycles in 13.7s on cuda (NVIDIA A100-SXM4-40GB)
[PASS] SMOKE g_P(0) > 0 and finite pipeline :: g=[ 3.5458e-03 -2.0133e-04 -7.1806e-05]

In [ ]:
# Cell 3 -- two-implementation cross-check, both streams on this GPU
import importlib.util, json as _json, sys as _sys, time as _time
import numpy as _np

G3 = dict(beta=5.99, L=8, Nt=16, thermal=400, ncfg=400, sep=4, ape=(0, 4))

# ---- stream A: reference CuPy Metropolis chain (your code, its own kernel) --
spec = importlib.util.spec_from_file_location("refmod", SCRIPT_REF)
refmod = importlib.util.module_from_spec(spec)
_sys.modules["refmod"] = refmod
_argv = _sys.argv; _sys.argv = ["refmod"]
spec.loader.exec_module(refmod)
_sys.argv = _argv

cfgA = refmod.EnsembleConfig(beta=G3["beta"], L=G3["L"], Nt=G3["Nt"],
                             thermal_cycles=G3["thermal"], n_cfg=G3["ncfg"],
                             separation_cycles=G3["sep"], overrelax_per_cycle=2,
                             ape_levels=G3["ape"], loop_shapes=("P",),
                             prefer_gpu=True, cold_start=True, seed=777)
BA = refmod.Backend(prefer_gpu=True, seed=777)
latA = refmod.SU3WilsonLattice(cfgA, BA)
t0 = _time.time()
for _ in range(cfgA.thermal_cycles):
    latA.cycle()
plaqA, obsA = [], []
for _ in range(cfgA.n_cfg):
    for _ in range(cfgA.separation_cycles):
        latA.cycle()
    plaqA.append(latA.plaquette())
    t1, _p = latA.measure_multiscale()
    obsA.append(t1)                       # [2 (P lvl0, P lvl4), 3, Nt]
obsA, plaqA = _np.stack(obsA), _np.asarray(plaqA)
print(f"stream A (CuPy Metropolis): done in {_time.time()-t0:.0f}s, "
      f"plaq = {plaqA.mean():.5f} +- {plaqA.std(ddof=1)/len(plaqA)**0.5:.5f}")

# ---- stream B: PyTorch heatbath chain (Cell 2 classes, its own kernel) ------
latB, obsB, plaqB = run_ensemble(G3["beta"], G3["L"], G3["Nt"], G3["thermal"],
                                 G3["ncfg"], G3["sep"], n_or=2,
                                 ape_levels=G3["ape"], label="stream B")
print(f"stream B (PyTorch heatbath): plaq = {plaqB.mean():.5f} "
      f"+- {plaqB.std(ddof=1)/len(plaqB)**0.5:.5f}")

# ---- shared statistics on both raw streams ----------------------------------
def analyse(obs, plaqs, chan_index, Nt):
    percfg = correlators(obs)[:, chan_index]
    tau = tau_int(percfg[:, 0]); block = max(2, int(math.ceil(2 * tau)))
    g, ge, m, me = blocked_bootstrap(percfg, Nt, 1, block)
    return dict(g=g, ge=ge, meff=m, merr=me, tau=tau,
                plaq=float(plaqs.mean()),
                plaq_err=float(plaqs.std(ddof=1) / len(plaqs) ** 0.5))

Nt = G3["Nt"]
A = {"P_lvl0": analyse(obsA, plaqA, 0, Nt), "P_lvl4": analyse(obsA, plaqA, 1, Nt)}
B = {"P_lvl0": analyse(obsB, plaqB, 0, Nt), "P_lvl4": analyse(obsB, plaqB, 2, Nt)}
# stream B channel order is [P0, P2_0, P4, P2_4]; P2 deferred pending
# the nextrun2 normalisation check (ImTr(W^2)/3 convention).

def sig(a, ea, b, eb): return abs(a - b) / math.sqrt(ea * ea + eb * eb + 1e-30)
npass = nfail = 0
def g3gate(name, s):
    global npass, nfail
    ok = s < 3.0; npass, nfail = npass + ok, nfail + (not ok)
    print(("[PASS] " if ok else "[FAIL] ") + f"{name}: |Delta|/sigma = {s:.2f}")

print("\n=== G3 gate table: CuPy Metropolis vs PyTorch heatbath (same GPU) ===")
g3gate("plaquette", sig(A["P_lvl0"]["plaq"], A["P_lvl0"]["plaq_err"],
                        B["P_lvl0"]["plaq"], B["P_lvl0"]["plaq_err"]))
for ch in ("P_lvl0", "P_lvl4"):
    for t in range(min(5, len(A[ch]["g"]))):
        g3gate(f"g_{ch}(t={t})", sig(A[ch]["g"][t], A[ch]["ge"][t],
                                     B[ch]["g"][t], B[ch]["ge"][t]))
    g3gate(f"meff_{ch}(1)", sig(A[ch]["meff"], A[ch]["merr"],
                                B[ch]["meff"], B[ch]["merr"]))
print(f"\nG3 SUMMARY: {npass}/{npass + nfail} gates passed")

dump = {"config": G3, "stream_A_cupy_metropolis":
        {k: {kk: (vv.tolist() if hasattr(vv, "tolist") else vv) for kk, vv in v.items()} for k, v in A.items()},
        "stream_B_pytorch_heatbath":
        {k: {kk: (vv.tolist() if hasattr(vv, "tolist") else vv) for kk, vv in v.items()} for k, v in B.items()}}
with open(os.path.join(OUTDIR, "su3_g3_crosscheck_b599.json"), "w") as f:
    _json.dump(dump, f, indent=1)
print("-> su3_g3_crosscheck_b599.json written to", OUTDIR)

stream A (CuPy Metropolis): done in 327s, plaq = 0.59312 +- 0.00008
  [stream B] beta=5.99 8^3x16: 400+400x4 cycles in 346.5s on cuda (NVIDIA A100-SXM4-40GB)
stream B (PyTorch heatbath): plaq = 0.59279 +- 0.00008

=== G3 gate table: CuPy Metropolis vs PyTorch heatbath (same GPU) ===
[PASS] plaquette: |Delta|/sigma = 2.99
[PASS] g_P_lvl0(t=0): |Delta|/sigma = 1.13
[PASS] g_P_lvl0(t=1): |Delta|/sigma = 0.48
[PASS] g_P_lvl0(t=2): |Delta|/sigma = 1.24
[PASS] g_P_lvl0(t=3): |Delta|/sigma = 1.74
[PASS] g_P_lvl0(t=4): |Delta|/sigma = 0.34
[FAIL] meff_P_lvl0(1): |Delta|/sigma = nan
[PASS] g_P_lvl4(t=0): |Delta|/sigma = 0.24
[PASS] g_P_lvl4(t=1): |Delta|/sigma = 0.74
[PASS] g_P_lvl4(t=2): |Delta|/sigma = 0.68
[PASS] g_P_lvl4(t=3): |Delta|/sigma = 1.35
[PASS] g_P_lvl4(t=4): |Delta|/sigma = 0.19
[FAIL] meff_P_lvl4(1): |Delta|/sigma = nan

G3 SUMMARY: 11/13 gates passed
-> su3_g3_crosscheck_b599.json written to /content/drive/MyDrive/su3_stage1


In [ ]:
# Cell 4 -- stage-1 production launcher (ONE ensemble per session)
ENSEMBLE_INDEX = 1        # 1 -> beta 5.99 (18^3x18), 2 -> 6.0625 (20^3x20), 3 -> 6.235 (26^3x26)
import os, subprocess, sys
json_out = os.path.join(OUTDIR, f"beta{ENSEMBLE_INDEX}.json")
cmd = [sys.executable, SCRIPT_PROD, "--profile", "continuum",
       "--ensemble", str(ENSEMBLE_INDEX), "--install-cupy", "--json", json_out]
print("launching:", " ".join(cmd))
print("Reminder: nextrun (non-2) has no mid-run checkpoint; calibrate on index 1 first.")
subprocess.run(cmd, check=True)
print("done ->", json_out)

launching: /usr/bin/python3 /content/ENGINE_MC_su3_t1pm_spatial_nextrun.py --profile continuum --ensemble 1 --install-cupy --json /content/drive/MyDrive/su3_stage1/beta1.json
Reminder: nextrun (non-2) has no mid-run checkpoint; calibrate on index 1 first.


### After all three ensemble JSONs exist

Combine and refit the continuum line in a final short session:

```
python SU3_T1pm_spatial_MC_nextrun2.py --profile combine \
    --inputs beta1.json beta2.json beta3.json
```

That combined fit is what updates the Section 5 replay-gate row and settles
the 6.065(40) vs 6.0598(441) fit-window question flagged in review. The
`su3_g3_crosscheck_b599.json` from Cell 3 is the two-implementation
certificate for the Monte Carlo codebase — cite it alongside the exact
certificate stack when the manuscript ships.